# 🪰 FlySpeak: Biological Drosophila Connectome Neural Language Model
### Training a Fruit Fly Brain Subcircuit to Predict and Generate Basic English Phrases

**Authors:** Arena Machine Learning Engineer  
**Hardware Support:** Kaggle Free GPU Tier (T4/P100) & CPU-only environments  
**Dataset:** MaleCNS v1.0 Fruit Fly Connectome (HHMI Janelia FlyEM / Google Research / Cambridge University)  
**Source URL:** [https://github.com/Kaos599/fly-brain-minesweeper/blob/main/data/malecns_circuit.json](https://github.com/Kaos599/fly-brain-minesweeper/blob/main/data/malecns_circuit.json)

---
## 1. Overview & Biological Motivation
Can a neural network constrained by the synaptic wiring of a fruit fly (*Drosophila melanogaster*) brain learn basic human conversational language?

In biological brains, neural computation is governed by:
1. **Sparse Synaptic Architecture:** Connections are not fully dense matrices; instead, they follow small-world, modular wiring diagrams.
2. **Dale's Principle:** Neurons release specific neurotransmitters that make their outgoing synapses strictly excitatory (e.g. Acetylcholine, $+1$) or inhibitory (e.g. GABA / Glutamate, hBc1$).
3. **Leaky Membrane Integration:** Biological neurons integrate synaptic currents over time with continuous membrane potential dynamics.

This notebook trains a **Connectome-Informed Recurrent Neural Network (FlyConnectomeRNN)** that embeds words into sensory neurons, propagates activations strictly through measured electron-microscopy biological synapses, and decodes conversational phrases from descending motor neurons.

In [ ]:
# Verify environment and available GPU/CPU acceleration
import torch
import os
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print(f"Running on CPU ({os.cpu_count()} cores)")

In [ ]:
# Run the standalone biological training pipeline script
!python train_fly_language.py --compare_baseline

## 2. Interactive Testing & Generation
Now that the model has trained, let us load the saved checkpoint () and interactively test novel conversational prompts!

In [ ]:
import torch
from train_fly_language import FlyConnectomeRNN, load_connectome_dataset, build_connectome_tensors, VOCABULARY

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
circuit_data = load_connectome_dataset("data/malecns_circuit.json")
bio_adj, bio_mask, meta = build_connectome_tensors(circuit_data)

model = FlyConnectomeRNN(len(VOCABULARY), meta["num_neurons"], bio_adj, bio_mask)
checkpoint = torch.load("fly_connectome_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print("Loaded trained FlyConnectomeRNN model successfully!")

test_prompts = ["hello", "how are", "thank", "good morning", "see you", "can i", "nice to"]
for p in test_prompts:
    out = model.generate(p, max_tokens=4, greedy=True)
    print(f"{p:<15} -> {out}")